# NatureCubePy Visualization Tutorial

This notebook demonstrates every public plotting function in `naturecubepy.viz`, grouped by plot family:

1. **Maps and station activity** — `station_map`, `station_explorer`, `camera_activity_timeline`
2. **Records and species summaries** — `records_per_class`, `plot_top_species`, `plot_top_species_by_sensor`
3. **Species accumulation** — `plot_species_accumulation_mammal_bird_by_sensor`
4. **eDNA and IUCN** — `plot_edna_records`, `plot_edna_unique_taxa_stacked`, `iucn_bar_plot`


In [ ]:
import matplotlib.pyplot as plt

from naturecubepy import (
    auth_headers,
    get_audio_observation_data,
    get_camera_trap_data,
    get_edna_observation_data,
    get_key,
    get_project_boundary,
    get_station_info,
)
from naturecubepy.viz import (
    camera_activity_timeline,
    iucn_bar_plot,
    plot_edna_records,
    plot_edna_unique_taxa_stacked,
    plot_species_accumulation_mammal_bird_by_sensor,
    plot_top_species,
    plot_top_species_by_sensor,
    records_per_class,
    station_explorer,
    station_map,
)


In [ ]:
# Retrieve API key and set up authentication headers
api_key = get_key("SL_GER_SIT")
hdr = auth_headers(api_key, "http://127.0.0.1:8000/api/")


## Load observation data

Pull stations plus camera, bioacoustic, and eDNA observations used by the plotters below.


In [ ]:
stations = get_station_info(hdr, measurement_type="all")
camtrap = get_camera_trap_data(hdr, include_iucn_status=True)
audio = get_audio_observation_data(hdr, include_iucn_status=True)
edna = get_edna_observation_data(hdr, include_iucn_status=True)

print(f"Stations: {len(stations)}")
print(f"Camera trap rows: {len(camtrap)}")
print(f"Audio rows: {len(audio)}")
print(f"eDNA rows: {len(edna)}")

stations.head()


## 1. Maps and station activity

Plot sampling locations as a static satellite map, an interactive Folium map, and deployment activity timelines.


### `station_map`

Static PNG-ready map of sampling locations. Marker size scales with record count; color encodes measurement type. Optionally overlay the project boundary.


In [ ]:
boundary = get_project_boundary(hdr)

fig = station_map(
    stations,
    measurement_type="all",
    project_boundary=boundary,
)
fig


In [ ]:
# Sensor-specific static maps
for sensor in ["Camera", "Bioacoustic", "eDNA"]:
    fig = station_map(stations, measurement_type=sensor, project_boundary=boundary)
    fig.suptitle(f"{sensor} sampling locations")
    display(fig)
    plt.close(fig)


### `station_explorer`

Interactive Folium map with popups for device metadata and media counts.


In [ ]:
station_explorer(stations, measurement_type="all")


### `camera_activity_timeline`

Deployment windows for each device. Works for camera or bioacoustic stations when start/end timestamp columns are present.


In [ ]:
camera_stations = stations[stations["measurement_type"] == "Camera"].copy()
fig, timeline_df = camera_activity_timeline(camera_stations)
print(f"Camera timeline rows: {len(timeline_df)}")
fig


In [ ]:
bio_stations = stations[stations["measurement_type"] == "Bioacoustic"].copy()
fig, timeline_df = camera_activity_timeline(bio_stations)
print(f"Bioacoustic timeline rows: {len(timeline_df)}")
fig


## 2. Records and species summaries

Summarise detection volume and the most recorded species by taxonomic class and sensor.


### `records_per_class`

Bar panels for number of records and/or number of species per taxonomic class. Use `panel="both"`, `"records"`, or `"species"`.


In [ ]:
camera_classes = sorted(camtrap["class"].dropna().astype(str).unique())
fig = records_per_class(camtrap, class_levels=camera_classes, panel="both")
fig


In [ ]:
audio_classes = sorted(audio["class"].dropna().astype(str).unique())
fig = records_per_class(audio, class_levels=audio_classes, panel="species")
fig


### `plot_top_species`

Horizontal bar charts of the top *N* species for each class in the supplied color map.


In [ ]:
fig = plot_top_species(camtrap, top_n=10)
fig


In [ ]:
# Restrict to selected classes / colors
fig = plot_top_species(
    camtrap,
    classes={"Mammalia": "#4C72B0", "Aves": "#55A868"},
    top_n=8,
)
fig


### `plot_top_species_by_sensor`

Convenience wrapper that returns mammal/bird top-species figures for camera and bioacoustic data.


In [ ]:
top_figs = plot_top_species_by_sensor(camtrap, audio, top_n=10)

for sensor, fig in top_figs.items():
    fig.suptitle(f"Top mammal and bird species — {sensor}")
    display(fig)
    plt.close(fig)


## 3. Species accumulation

Incidence-based mammal vs bird accumulation curves by sensor effort (camera-trap days / bioacoustic days).


### `plot_species_accumulation_mammal_bird_by_sensor`


In [ ]:
accumulation_figs = plot_species_accumulation_mammal_bird_by_sensor(
    camtrap,
    audio,
    stations,
)

for sensor, fig in accumulation_figs.items():
    fig.suptitle(f"Species accumulation — {sensor}")
    display(fig)
    plt.close(fig)


## 4. eDNA and IUCN plots

Stacked identification-level charts for eDNA taxa, plus IUCN red-list status bars.


### `plot_edna_records`

Stacked bars of unique taxa by taxonomic class, colored by lowest identification rank (Order / Family / Genus / Species).


In [ ]:
fig, edna_summary = plot_edna_records(edna, return_summary=True)
display(edna_summary)
fig


### `plot_edna_unique_taxa_stacked`

Legacy alias for `plot_edna_records` — same figure, kept for older report call sites.


In [ ]:
fig = plot_edna_unique_taxa_stacked(edna)
fig


### `iucn_bar_plot`

Stacked bar chart of unique species by taxonomic class and IUCN status. Pass the IUCN column name used in your observation frame — typically `iucn_redlist_status`.


In [ ]:
# IUCN status from eDNA observations
fig = iucn_bar_plot(
    edna,
    class_col="class",
    species_col="species",
    iucn_col="iucn_redlist_status",
)
fig


In [ ]:
# Same plotter works on camera-trap rows with IUCN enrichment
fig, iucn_summary = iucn_bar_plot(
    camtrap,
    class_col="class",
    species_col="species",
    iucn_col="iucn_redlist_status",
    return_summary=True,
)
display(iucn_summary)
fig
